# Practical-8: Course Review Sentiment Classifier (Multinomial Naive Bayes)

**Aim:** To classify student course reviews as Positive, Negative or Neutral using Multinomial Naive Bayes, and discuss ethics and explainability.

In [1]:
import os
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [2]:
# Load the review dataset
# Place "student_course_reviews_500.csv" (columns: review_text, sentiment) next to this notebook.
# If the file is not found, a small synthetic dataset is generated so the notebook still runs.
CSV_PATH = "student_course_reviews_500.csv"

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    print("CSV not found - generating a synthetic dataset of 500 reviews.")
    random.seed(42)
    pos = ["The course was useful and the instructor explained concepts clearly.",
           "Excellent teaching, great examples and very helpful assignments.",
           "I loved this course, the content was engaging and well organised.",
           "Very informative lectures and the faculty was supportive."]
    neg = ["The course was boring and the lectures were confusing.",
           "Poorly organised, too many assignments and unclear explanations.",
           "I did not learn much, the instructor was not helpful.",
           "Terrible pace and the content felt outdated."]
    neu = ["The course was okay, neither good nor bad.",
           "Average lectures with some useful and some dull topics.",
           "It was a standard course with a normal workload.",
           "Some parts were fine while others could be improved."]
    extra = ["Overall", "Honestly", "In my opinion", "To be fair", "Personally"]
    rows = []
    for _ in range(500):
        label, pool = random.choice([("Positive", pos), ("Negative", neg), ("Neutral", neu)])
        rows.append({"review_text": random.choice(extra) + ", " + random.choice(pool).lower(),
                     "sentiment": label})
    df = pd.DataFrame(rows)

df.head()

CSV not found - generating a synthetic dataset of 500 reviews.


,review_text,sentiment
0,"Overall, the course was okay, neither good nor...",Neutral
1,"In my opinion, average lectures with some usef...",Neutral
2,"Honestly, the course was useful and the instru...",Positive
3,"Personally, the course was okay, neither good ...",Neutral
4,"To be fair, the course was okay, neither good ...",Neutral


In [3]:
# Remove rows with missing review text or sentiment labels
df_clean = df.dropna()

# Separate review text (input) and sentiment (target)
x = df_clean["review_text"]
y = df_clean["sentiment"]

# Split data while preserving class proportions
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)

# Convert text into word-count features
vectorizer = CountVectorizer()
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

# Train the Multinomial Naive Bayes classifier
model = MultinomialNB()
model.fit(x_train_vec, y_train)

# Predict sentiment for test reviews and evaluate the model
y_pred = model.predict(x_test_vec)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Example prediction for a new review (optional)
new_review = ["The course was useful and the instructor explained concepts clearly."]
new_review_vec = vectorizer.transform(new_review)
print("Example review sentiment:", model.predict(new_review_vec)[0])

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        53
     Neutral       1.00      1.00      1.00        49
    Positive       1.00      1.00      1.00        48

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

Example review sentiment: Positive


In [4]:
# Explainability: top words influencing each sentiment class
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
for i, cls in enumerate(model.classes_):
    top = np.argsort(model.feature_log_prob_[i])[-8:][::-1]
    print(f"{cls}: {', '.join(feature_names[top])}")

Negative: the, and, was, not, too, many, organised, explanations
Neutral: some, be, was, course, with, the, neither, bad
Positive: the, and, was, course, very, this, well, content
